# MVA Hackathon 2026 — Track 2 · fault-tree / FMEA triage of repurposing hypotheses (yasld)
Runs without patient data: it reads a curated knowledge file (`knowledge/*.json`). Optional last step: if the Track 1
output files `nmd_check.json` and `locus_summary.json` are present in `TRACK1_WORK`, the nonsense allele is checked for
translational-readthrough context (needs Internet for two reference bases from the UCSC API).
Outputs are **hypotheses for follow-up, not treatment advice**. Severity and detection scores are expert judgements; see the rank-stability table.

In [ ]:
!pip install -q pandas matplotlib numpy
import os, sys, json
sys.path.insert(0, os.getcwd())
os.makedirs("mvapipe/track2", exist_ok=True); os.makedirs("knowledge", exist_ok=True)
open("mvapipe/__init__.py", "a").close()

In [ ]:
%%writefile mvapipe/track2/__init__.py
"""Track 2 (drug repurposing): fault-tree + FMEA triage of intervention hypotheses.

Cross-sector methods: fault-tree analysis (Bell Labs, 1960s; aerospace and nuclear safety) maps how a
genotype fails; failure-mode-and-effects analysis (US military and NASA origins; automotive industry) ranks
interventions by risk, with hard stops. Outputs are hypotheses for follow-up, never treatment advice.
"""

In [ ]:
%%writefile mvapipe/track2/faulttree.py
"""Minimal fault-tree engine (AND / OR gates over basic events) with minimal cut sets and coverage.

A cut set is a set of basic events whose joint occurrence produces the top event. An intervention that
clears basic event(s) removes every cut set that contains them. 'Coverage' of an intervention is the
fraction of minimal cut sets it breaks; an intervention that breaks all of them fully mitigates the tree.
"""
from __future__ import annotations

from itertools import product


def _minimise(sets: list[frozenset]) -> list[frozenset]:
    sets = sorted(set(sets), key=len)
    out: list[frozenset] = []
    for s in sets:
        if not any(o <= s for o in out):
            out.append(s)
    return out


def cut_sets(tree: dict, node: str | None = None) -> list[frozenset]:
    """tree = {"top": id, "nodes": {id: {"type": "AND"|"OR"|"BASIC", "children": [ids], "label": str}}}"""
    node = node or tree["top"]
    spec = tree["nodes"][node]
    kind = spec["type"]
    if kind == "BASIC":
        return [frozenset([node])]
    child = [cut_sets(tree, c) for c in spec["children"]]
    if kind == "OR":
        return _minimise([s for c in child for s in c])
    if kind == "AND":
        combos = [frozenset().union(*pick) for pick in product(*child)]
        return _minimise(combos)
    raise ValueError(f"unknown gate type {kind!r}")


def coverage(cuts: list[frozenset], cleared: set[str]) -> float:
    """Fraction of minimal cut sets broken when the basic events in `cleared` are removed."""
    if not cuts:
        return 0.0
    return sum(1 for c in cuts if c & cleared) / len(cuts)


def fully_mitigating_sets(cuts: list[frozenset]) -> list[frozenset]:
    """Smallest sets of basic events that hit every cut set (minimal hitting sets), by brute force.
    Trees here are tiny, so exhaustive search is fine."""
    events = sorted(set().union(*cuts)) if cuts else []
    best: list[frozenset] = []
    for r in range(1, len(events) + 1):
        for comb in __import__("itertools").combinations(events, r):
            cs = frozenset(comb)
            if all(c & cs for c in cuts) and not any(b <= cs for b in best):
                best.append(cs)
        if best:
            break
    return best


def validate(tree: dict) -> list[str]:
    problems = []
    nodes = tree.get("nodes", {})
    if tree.get("top") not in nodes:
        problems.append("top node missing")
    for k, v in nodes.items():
        if v["type"] in ("AND", "OR"):
            for c in v.get("children", []):
                if c not in nodes:
                    problems.append(f"{k}: unknown child {c}")
        elif v["type"] != "BASIC":
            problems.append(f"{k}: bad type {v['type']}")
    return problems

In [ ]:
%%writefile mvapipe/track2/fmea.py
"""FMEA-style triage of intervention hypotheses, with hard stops and a rank-stability analysis.

Scales (1-10, as in classical FMEA; lower RPN is better):
  S  severity   : harm to the patient if the intervention is given and the hypothesis is wrong.
  O  occurrence : likelihood that the hypothesis is wrong or ineffective. Derived, not typed in:
                  O = round(10 - 1.2*E - 2*c), where E is the evidence grade (1-5) and c the coverage (0-1).
  D  detection  : difficulty of noticing failure or harm in time (1 = a validated biomarker exists).
  RPN = S * O * D.

Hard stops (applied before ranking): evidence grade below 2; no approved route (availability); severity >= 9.
The S and D values are expert judgements stored with a rationale in the knowledge file; the rank-stability
analysis shows how far they can move before the order changes.
"""
from __future__ import annotations

import numpy as np

E_MIN = 2
S_STOP = 9
APPROVED = ("approved_paediatric", "approved_adult", "approved_off_label")


def clamp(x: float, lo: int = 1, hi: int = 10) -> int:
    return int(max(lo, min(hi, round(x))))


def occurrence(evidence: int, coverage: float) -> int:
    return clamp(10 - 1.2 * evidence - 2.0 * coverage)


def hard_stops(item: dict) -> list[str]:
    """All hard-stop reasons that apply (an item can fail on several grounds)."""
    why = []
    if item["E"] < E_MIN:
        why.append(f"insufficient evidence (grade {item['E']} < {E_MIN})")
    if item.get("approval", "none") not in APPROVED + ("supplement",):
        why.append("no approved or available agent")
    if item["S"] >= S_STOP:
        why.append(f"severity {item['S']} >= {S_STOP}")
    if item.get("opposite_mechanism"):
        why.append("acts in the opposite direction to the mechanism (" + item["opposite_mechanism"] + ")")
    return why


def hard_stop(item: dict) -> str | None:
    w = hard_stops(item)
    return "; ".join(w) if w else None


def score(item: dict, cov: float) -> dict:
    o = occurrence(item["E"], cov)
    return {"id": item["id"], "label": item["label"], "tier": item["tier"], "coverage": round(cov, 2),
            "E": item["E"], "S": item["S"], "O": o, "D": item["D"], "RPN": item["S"] * o * item["D"],
            "stop": hard_stop(item)}


def rank(rows: list[dict]) -> tuple[list[dict], list[dict]]:
    kept = sorted([r for r in rows if r["stop"] is None], key=lambda r: (r["RPN"], r["S"]))
    out = [r for r in rows if r["stop"] is not None]
    return kept, out


def rank_stability(rows: list[dict], n: int = 10000, jitter: int = 1, seed: int = 2026) -> dict:
    """Perturb S, O and D of every candidate by an integer in [-jitter, +jitter] and recompute the order."""
    rng = np.random.default_rng(seed)
    ids = [r["id"] for r in rows]
    base = np.array([[r["S"], r["O"], r["D"]] for r in rows], dtype=float)
    ranks = np.zeros((n, len(rows)), dtype=int)
    same = 0
    base_order = list(np.argsort([r["RPN"] for r in rows], kind="stable"))
    for k in range(n):
        noise = rng.integers(-jitter, jitter + 1, size=base.shape)
        v = np.clip(base + noise, 1, 10)
        rpn = v[:, 0] * v[:, 1] * v[:, 2]
        order = list(np.argsort(rpn, kind="stable"))
        for pos, idx in enumerate(order):
            ranks[k, idx] = pos + 1
        same += order == base_order
    return {
        "n": n, "jitter": jitter, "p_same_order": same / n,
        "per_candidate": {i: {"mean_rank": float(ranks[:, j].mean()),
                              "p_rank1": float((ranks[:, j] == 1).mean()),
                              "p_same_rank_as_base": float((ranks[:, j] == (base_order.index(j) + 1)).mean())}
                          for j, i in enumerate(ids)},
    }

In [ ]:
%%writefile mvapipe/track2/readthrough.py
"""Variant-specific check of eligibility for translational readthrough (nonsense variants only).

What is computed: which stop codon the variant creates, the base at +4 (immediately 3' of the codon), the
qualitative readthrough propensity from the published hierarchies (UGA > UAG > UAA; a C at +4 favours
readthrough, a G disfavours it), and whether nonsense-mediated decay is predicted to remove the transcript
first. These hierarchies are population-level tendencies, not predictions for a given allele.
"""
from __future__ import annotations

import itertools
import json
import urllib.request

BASES = "TCAG"
AA = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"
CODON = {a + b + c: AA[i] for i, (a, b, c) in enumerate(itertools.product(BASES, repeat=3))}
THREE = {"Ala": "A", "Arg": "R", "Asn": "N", "Asp": "D", "Cys": "C", "Gln": "Q", "Glu": "E", "Gly": "G", "His": "H",
         "Ile": "I", "Leu": "L", "Lys": "K", "Met": "M", "Phe": "F", "Pro": "P", "Ser": "S", "Thr": "T",
         "Trp": "W", "Tyr": "Y", "Val": "V"}
COMP = str.maketrans("ACGT", "TGCA")


def candidate_stop_codons(ref_aa3: str, ref: str, alt: str, strand: str) -> list[dict]:
    """All codons for amino acid `ref_aa3` in which substituting `ref`>`alt` (genomic, given strand) makes a
    stop. Returns codon, index of the changed base in the codon (0-2, 5'->3' on the transcript) and the stop."""
    aa = THREE[ref_aa3]
    r, a = (ref, alt) if strand == "+" else (ref.translate(COMP), alt.translate(COMP))
    out = []
    for cod, x in CODON.items():
        if x != aa:
            continue
        for i in range(3):
            if cod[i] == r:
                new = cod[:i] + a + cod[i + 1:]
                if CODON[new] == "*":
                    out.append({"codon": cod, "index": i, "stop": new.replace("T", "U")})
    return out


def plus4_coordinate(pos: int, index: int, strand: str) -> int:
    """Genomic (1-based) coordinate of the base at +4, i.e. 3' of the stop codon on the transcript."""
    return pos - index + 3 if strand == "+" else pos + index - 3


def minus1_coordinate(pos: int, index: int, strand: str) -> int:
    """Genomic coordinate of the base immediately 5' of the stop codon on the transcript (position -1)."""
    return pos - index - 1 if strand == "+" else pos + index + 1


def propensity(stop: str, plus4: str | None, minus1: str | None = None) -> dict:
    """Heuristic, additive score from the context features reported by Floquet et al. 2012 (PLoS Genet):
    basal readthrough UGA > UAG > UAA; a C at +4 raises readthrough; a U at -1 is the main determinant of the
    gentamicin response. It orders contexts qualitatively and is not a prediction of efficacy."""
    s = {"UGA": 2, "UAG": 1, "UAA": 0}[stop]
    p = {"C": 1, "U": 0, "T": 0, "A": 0, "G": -1}.get(plus4 or "", 0)
    m = 1 if (minus1 or "") in ("T", "U") else 0
    total = s + p + m
    level = "higher" if total >= 3 else "intermediate" if total >= 1 else "lower"
    return {"stop_score": s, "plus4_score": p, "minus1_score": m, "level": level}


def fetch_base(chrom: str, pos: int, genome: str = "hg38", timeout: int = 20) -> str:
    """One reference base from the UCSC sequence API (needs internet). Returns upper-case A/C/G/T."""
    c = chrom if str(chrom).startswith("chr") else f"chr{chrom}"
    url = f"https://api.genome.ucsc.edu/getData/sequence?genome={genome};chrom={c};start={pos - 1};end={pos}"
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return json.load(r)["dna"].upper()


def assess(ref_aa3: str, ref: str, alt: str, strand: str, nmd_predicted: bool,
           plus4_base: str | None = None, minus1_base: str | None = None) -> dict:
    cands = candidate_stop_codons(ref_aa3, ref, alt, strand)
    res = {"candidates": cands, "unique": len(cands) == 1, "nmd_predicted": bool(nmd_predicted)}
    if len(cands) != 1:
        res["verdict"] = "stop codon identity ambiguous from the amino-acid change alone"
        return res
    c = cands[0]
    plus4 = plus4_base.upper() if plus4_base else None
    if plus4 and strand == "-":
        plus4 = plus4.translate(COMP)
    minus1 = minus1_base.upper() if minus1_base else None
    if minus1 and strand == "-":
        minus1 = minus1.translate(COMP)
    prop = propensity(c["stop"], plus4, minus1)
    res.update({"stop": c["stop"], "plus4": plus4, "minus1": minus1, "propensity": prop})
    if nmd_predicted:
        res["verdict"] = ("limited: the transcript is predicted to be degraded by NMD before translation, so "
                          "readthrough alone cannot restore expression (no approved NMD inhibitor)")
    else:
        res["verdict"] = f"candidate for readthrough ({prop['level']} propensity)"
    return res

In [ ]:
%%writefile mvapipe/track2/scenarios.py
"""Genotype-uncertainty analysis.

The fault tree assumes the two BUB1B alleles are in trans and that the missense allele is hypomorphic. Track 1 could
establish neither (unphased short reads; the missense is a variant of uncertain significance). Let

    r = P(trans) * P(missense hypomorphic)

be the probability that the compound-heterozygous genotype is the disease-causing one. For root-tier interventions
(which act on an allele) the effective coverage is c * r: if the genotype is not causal they have nothing to repair.
Consequence-tier interventions act on harm that is present whatever the cause, so their coverage is unchanged
(they remain gated by a biomarker). An optional modifier (a second tumour-predisposition variant) raises the severity
of interventions tagged with a matching risk.

Outputs: the order of candidates over a grid of r, the threshold at which the order changes, and a minimax view
(worst RPN over the grid).
"""
from __future__ import annotations

from . import faulttree as ft
from . import fmea


def score_under(item: dict, cuts: dict, r: float, modifier: dict | None = None) -> dict:
    """Score one candidate when the compound-heterozygous genotype is causal with probability r.

    Failure-probability proxy f = (10 - 1.2 E - 2 c) / 10 as in the base model. A root-tier hypothesis can only work if
    the genotype is causal, so f_eff = 1 - (1 - f) * r; consequence-tier hypotheses keep f. O = clamp(round(10 f_eff)).
    """
    cov = ft.coverage(cuts[item["tier"]], set(item["addresses"]))
    f = (10 - 1.2 * item["E"] - 2.0 * cov) / 10
    f_eff = 1 - (1 - f) * r if item["tier"] == "root" else f
    it = dict(item)
    if modifier:
        d = max([modifier["severity_delta"].get(t, 0) for t in item.get("tags", [])] + [0])
        it["S"] = min(10, item["S"] + d)
    o = fmea.clamp(10 * f_eff)
    row = {"id": it["id"], "label": it["label"], "tier": it["tier"], "coverage": round(cov, 2), "E": it["E"],
           "S": it["S"], "O": o, "D": it["D"], "RPN": it["S"] * o * it["D"], "stop": fmea.hard_stop(it), "r": r}
    return row


def scenario_table(kb: dict, use_modifier: bool = True) -> dict:
    u = kb.get("uncertainty", {})
    grid = u.get("r_grid", [0.2, 0.5, 0.8, 1.0])
    mod = u.get("modifier") if use_modifier else None
    cuts = {"root": ft.cut_sets(kb["root_tree"]), "consequence": ft.cut_sets(kb["consequence_tree"])}
    rows, orders = [], {}
    for r in grid:
        sc = [score_under(it, cuts, r, mod) for it in kb["interventions"]]
        kept, _ = fmea.rank(sc)
        orders[r] = [x["id"] for x in kept]
        rows += sc
    ids = [it["id"] for it in kb["interventions"]]
    worst = {i: max(x["RPN"] for x in rows if x["id"] == i) for i in ids if any(x["id"] == i and x["stop"] is None for x in rows)}
    best = {i: min(x["RPN"] for x in rows if x["id"] == i) for i in worst}
    changes = [(a, b) for a, b in zip(grid, grid[1:]) if orders[a] != orders[b]]
    return {"r_grid": grid, "orders": {str(k): v for k, v in orders.items()}, "order_changes_between_r": changes,
            "rpn_worst_over_grid": worst, "rpn_best_over_grid": best,
            "minimax_order": sorted(worst, key=worst.get), "modifier_applied": bool(mod), "rows": rows}

In [ ]:
%%writefile mvapipe/track2/checkpoint_sim.py
"""Qualitative, parameter-free-of-data simulation of the spindle assembly checkpoint (SAC) with BUBR1 dosage.

THIS IS A TOY MODEL, NOT A PREDICTOR. Its parameters are not fitted to data. What it is used for is a *direction-of-effect*
test: for broad, random parameter sets, does an intervention that acts on mitosis raise or lower chromosome
mis-segregation in a checkpoint-weakened cell? A conclusion is reported only if it holds across the parameter ranges.

Model (per cell, expected values). N chromosomes each start with an attachment error with probability e0; an error is
corrected at rate kc. Checkpoint signal while E(t) = N e0 exp(-kc t) errors remain:  M(t) = g(b) * (1 - m) * E(t),
with g(b) = b**h (b = BUBR1 level relative to normal, h = dosage sensitivity) and m = fractional MPS1 inhibition.
Anaphase starts when M(t) < theta (checkpoint satisfied) but not before a fixed minimum time T0, and the cell slips out of
mitosis at T_max whatever the state. Mis-segregation = expected number of uncorrected errors at anaphase onset.
Not modelled: DNA damage, cell death, spatial effects, the specific biochemistry of MCC assembly, tissues.
"""
from __future__ import annotations

import numpy as np

N_CHR = 46


def draw_params(n: int, seed: int = 2026) -> dict:
    rng = np.random.default_rng(seed)
    lu = lambda lo, hi: np.exp(rng.uniform(np.log(lo), np.log(hi), n))
    return {"e0": lu(0.03, 0.3), "kc": lu(0.02, 0.3), "theta": lu(0.1, 3.0), "T0": lu(5, 30),
            "Tmax": lu(100, 600), "h": rng.uniform(1.0, 3.0, n)}


def missegregation(p: dict, b, mps1_inhib=0.0, err_mult=1.0, corr_mult=1.0):
    """Expected uncorrected errors at anaphase, and mitotic duration, for each parameter set."""
    e0 = np.clip(p["e0"] * err_mult, 0, 1)
    kc = p["kc"] * corr_mult
    s = (np.asarray(b, dtype=float) ** p["h"]) * (1.0 - mps1_inhib)
    n_err0 = N_CHR * e0
    need = np.where(n_err0 * s > p["theta"], np.log(np.maximum(n_err0 * s, 1e-12) / p["theta"]) / kc, 0.0)
    T = np.minimum(np.maximum(p["T0"], need), p["Tmax"])
    return n_err0 * np.exp(-kc * T), T


# interventions on mitosis: (label, kwargs applied to the DISEASE state)
INTERVENTIONS = {
    "BUBR1 level +50% (I1-like)": {"b_mult": 1.5},
    "BUBR1 level +100% (I1-like, upper bound)": {"b_mult": 2.0},
    "MPS1 inhibitor, 30% inhibition": {"mps1_inhib": 0.3},
    "MPS1 inhibitor, 70% inhibition": {"mps1_inhib": 0.7},
    "Microtubule agent, low dose (X2-like)": {"err_mult": 1.5, "corr_mult": 0.7},
    "Microtubule agent, moderate dose (X2-like)": {"err_mult": 2.5, "corr_mult": 0.5},
}


def run(n: int = 5000, b_disease: float = 0.35, b_healthy: float = 1.0, seed: int = 2026) -> dict:
    p = draw_params(n, seed)
    base_d, T_d = missegregation(p, b_disease)
    base_h, T_h = missegregation(p, b_healthy)
    out = {"n_parameter_sets": n, "b_disease": b_disease, "model_note": "toy model, not fitted to data",
           "baseline": {"misseg_disease_median": float(np.median(base_d)), "misseg_healthy_median": float(np.median(base_h)),
                        "p_disease_worse_than_healthy": float(np.mean(base_d > base_h + 1e-9)),
                        "p_disease_not_better": float(np.mean(base_d >= base_h - 1e-9))},
           "interventions": {}}
    for name, k in INTERVENTIONS.items():
        bb = min(b_disease * k.get("b_mult", 1.0), 1.0)
        m_d, T = missegregation(p, bb, k.get("mps1_inhib", 0.0), k.get("err_mult", 1.0), k.get("corr_mult", 1.0))
        m_h, _ = missegregation(p, b_healthy, k.get("mps1_inhib", 0.0), k.get("err_mult", 1.0), k.get("corr_mult", 1.0))
        eps = 1e-9
        d_dis = m_d - base_d
        d_hea = m_h - base_h
        tied = np.abs(d_dis) <= eps
        out["interventions"][name] = {
            "p_worse": float(np.mean(d_dis > eps)), "p_better": float(np.mean(d_dis < -eps)), "p_no_change": float(np.mean(tied)),
            "median_change_disease": float(np.median(d_dis)),
            "median_relative_change_disease": float(np.median(d_dis / np.maximum(base_d, 1e-9))),
            "median_change_healthy": float(np.median(d_hea)),
            "p_disease_harmed_more_than_healthy": float(np.mean(d_dis > d_hea + eps)),
        }
    out["_raw"] = {"base": base_d, "params": p}
    return out


def dose_response(p: dict, bs=None) -> dict:
    bs = bs if bs is not None else np.linspace(0.1, 1.0, 10)
    return {float(b): float(np.median(missegregation(p, b)[0])) for b in bs}

In [ ]:
%%writefile mvapipe/track2/db_screen.py
"""Database screen for candidate drugs (run where Internet is available, e.g. Kaggle with Internet ON).

Queries three public resources for the gene sets of the mechanism (spindle checkpoint, interferon / cGAS-STING axis,
senescence, NAD+/sirtuin axis) and writes a table of drug-gene pairs plus a pathway-enrichment table:
  * DGIdb (GraphQL)            drug-gene interactions, approval flag
  * Open Targets (GraphQL)     clinical-stage drugs per target (field names have changed between releases; both tried)
  * Enrichr (REST)             pathway enrichment of the gene sets
Every request failure is recorded, never silently skipped. No patient data are used or sent: only gene symbols.
Usage:  python -m mvapipe.track2.db_screen --out OUTDIR
"""
from __future__ import annotations

import argparse
import json
import time
import urllib.parse
import urllib.request
from pathlib import Path

GENE_SETS = {
    "spindle_checkpoint": ["BUB1B", "BUB1", "BUB3", "MAD2L1", "MAD1L1", "CDC20", "TTK", "AURKB", "PPP2R5A", "PLK1", "KNL1"],
    "innate_immune_axis": ["CGAS", "STING1", "TBK1", "IRF3", "IFNAR1", "IFNAR2", "JAK1", "JAK2", "TYK2", "STAT1", "STAT2"],
    "senescence_sasp": ["CDKN2A", "CDKN1A", "TP53", "BCL2", "BCL2L1", "SRC", "IL6", "MMP3"],
    "nad_sirtuin": ["SIRT1", "SIRT2", "NAMPT", "NMNAT1", "NAPRT", "PARP1"],
}
UA = {"User-Agent": "mva-track2-screen/1.0", "Content-Type": "application/json"}


def _post(url: str, payload: dict, timeout: int = 60):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(), headers=UA)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.load(r)


def dgidb(genes: list[str]) -> list[dict]:
    q = """query($g:[String!]){genes(names:$g){nodes{name interactions{drug{name approved}
           interactionScore interactionTypes{type directionality}}}}}"""
    d = _post("https://dgidb.org/api/graphql", {"query": q, "variables": {"g": genes}})
    rows = []
    for g in d["data"]["genes"]["nodes"]:
        for it in g["interactions"]:
            rows.append({"source": "DGIdb", "gene": g["name"], "drug": it["drug"]["name"], "approved": it["drug"].get("approved"),
                         "type": ";".join(t["type"] or "" for t in it.get("interactionTypes", [])), "score": it.get("interactionScore")})
    return rows


def ensembl_id(symbol: str) -> str | None:
    q = 'query($q:String!){search(queryString:$q,entityNames:["target"]){hits{id name}}}'
    d = _post("https://api.platform.opentargets.org/api/v4/graphql", {"query": q, "variables": {"q": symbol}})
    for h in d["data"]["search"]["hits"]:
        if h["name"].upper() == symbol.upper():
            return h["id"]
    return None


def opentargets(symbol: str) -> list[dict]:
    eid = ensembl_id(symbol)
    if not eid:
        return []
    url = "https://api.platform.opentargets.org/api/v4/graphql"
    attempts = [
        ("drugAndClinicalCandidates", """query($id:String!){target(ensemblId:$id){drugAndClinicalCandidates{rows{maxClinicalStage drug{name}}}}}"""),
        ("knownDrugs", """query($id:String!){target(ensemblId:$id){knownDrugs{rows{phase status drug{name} disease{name}}}}}"""),
    ]
    last = None
    for name, q in attempts:
        try:
            d = _post(url, {"query": q, "variables": {"id": eid}})
            if d.get("errors"):
                last = d["errors"][0]["message"]; continue
            rows = d["data"]["target"][name]["rows"]
            return [{"source": "OpenTargets", "gene": symbol, "drug": r["drug"]["name"],
                     "stage": r.get("maxClinicalStage") or r.get("phase"), "disease": (r.get("disease") or {}).get("name")} for r in rows]
        except Exception as e:                       # record and try the next schema
            last = repr(e)
    raise RuntimeError(f"Open Targets: no schema variant worked ({last})")


def enrichr(genes: list[str], libs=("Reactome_2022", "GO_Biological_Process_2023", "KEGG_2021_Human")) -> list[dict]:
    boundary = "----mva"
    body = (f'--{boundary}\r\nContent-Disposition: form-data; name="list"\r\n\r\n' + "\n".join(genes) +
            f'\r\n--{boundary}\r\nContent-Disposition: form-data; name="description"\r\n\r\nmva\r\n--{boundary}--\r\n').encode()
    req = urllib.request.Request("https://maayanlab.cloud/Enrichr/addList", data=body,
                                 headers={"Content-Type": f"multipart/form-data; boundary={boundary}", "User-Agent": UA["User-Agent"]})
    uid = json.load(urllib.request.urlopen(req, timeout=60))["userListId"]
    rows = []
    for lib in libs:
        url = f"https://maayanlab.cloud/Enrichr/enrich?userListId={uid}&backgroundType={urllib.parse.quote(lib)}"
        d = json.load(urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": UA["User-Agent"]}), timeout=60))
        for r in d[lib][:8]:      # rank, term, p, z, combined, genes, adj_p, ...
            rows.append({"library": lib, "term": r[1], "p": r[2], "adj_p": r[6], "genes": ";".join(r[5])})
        time.sleep(0.5)
    return rows


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--out", default="work_track2")
    a = ap.parse_args(argv)
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    import pandas as pd
    errors, drugs, enr = [], [], []
    allg = sorted({g for v in GENE_SETS.values() for g in v})
    try:
        drugs += dgidb(allg)
    except Exception as e:
        errors.append(f"DGIdb: {e!r}")
    for g in allg:
        try:
            drugs += opentargets(g)
        except Exception as e:
            errors.append(f"OpenTargets {g}: {e!r}")
    for name, genes in GENE_SETS.items():
        try:
            enr += [{"set": name, **r} for r in enrichr(genes)]
        except Exception as e:
            errors.append(f"Enrichr {name}: {e!r}")
    pd.DataFrame(drugs).to_csv(out / "db_drug_gene.csv", index=False)
    pd.DataFrame(enr).to_csv(out / "db_enrichment.csv", index=False)
    (out / "db_errors.json").write_text(json.dumps(errors, indent=1))
    print("drug-gene rows:", len(drugs), "| enrichment rows:", len(enr), "| errors:", len(errors))
    for e in errors[:10]:
        print(" ", e)
    if drugs:
        df = pd.DataFrame(drugs)
        print(df.groupby("drug").agg(genes=("gene", lambda s: ",".join(sorted(set(s)))), n=("gene", "nunique"))
              .sort_values("n", ascending=False).head(40).to_string())
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile mvapipe/track2/run.py
"""Orchestrator for Track 2: fault-tree coverage, FMEA ranking, rank stability, readthrough check, figures.

    python -m mvapipe.track2.run [--kb knowledge/bub1b_mva1_kb.json] [--out DIR] [--no-network]

Reads only the curated knowledge file; optionally uses the Track 1 outputs (nmd_check.json,
locus_summary.json) for the variant-specific readthrough check. No coordinates are written to the outputs.
"""
from __future__ import annotations

import argparse
import json
import os
import re
from pathlib import Path

import pandas as pd

from . import faulttree as ft
from . import fmea
from . import readthrough as rt
from . import scenarios as scn
from . import checkpoint_sim as sim


def load_kb(path=None) -> dict:
    path = Path(path or os.environ.get("MVA_KB") or Path(__file__).resolve().parents[2] / "knowledge" / "bub1b_mva1_kb.json")
    with open(path) as fh:
        return json.load(fh)


def analyse(kb: dict) -> dict:
    root, cons = kb["root_tree"], kb["consequence_tree"]
    for t in (root, cons):
        problems = ft.validate(t)
        if problems:
            raise ValueError("; ".join(problems))
    cuts = {"root": ft.cut_sets(root), "consequence": ft.cut_sets(cons)}
    rows = []
    for it in kb["interventions"]:
        cov = ft.coverage(cuts[it["tier"]], set(it["addresses"]))
        rows.append(fmea.score(it, cov))
    kept, stopped = fmea.rank(rows)
    for ex in kb["exclusions"]:
        stopped.append({"id": ex["id"], "label": ex["label"], "tier": ex["class"], "E": ex["E"], "S": ex["S"],
                        "stop": fmea.hard_stop({"E": ex["E"], "S": ex["S"], "approval": ex["approval"],
                                                        "opposite_mechanism": ex.get("opposite_mechanism")})
                        or "excluded on mechanistic grounds", "reason": ex["reason"]})
    stab = fmea.rank_stability(kept) if len(kept) > 1 else {}
    return {
        "minimal_cut_sets": {k: sorted(sorted(c) for c in v) for k, v in cuts.items()},
        "fully_mitigating_sets_root": sorted(sorted(s) for s in ft.fully_mitigating_sets(cuts["root"])),
        "ranking": kept, "excluded": stopped, "rank_stability": stab,
        "scenarios": {m: {k: v for k, v in scn.scenario_table(kb, m == "with_modifier").items() if k != "rows"}
                      for m in ("without_modifier", "with_modifier")} if "uncertainty" in kb else {},
    }


def readthrough_check(workdir: Path, network: bool = True) -> dict | None:
    nmd_f, loc_f = workdir / "nmd_check.json", workdir / "locus_summary.json"
    if not (nmd_f.exists() and loc_f.exists()):
        return None
    nmd = json.load(open(nmd_f))["results"]
    strand = json.load(open(loc_f)).get("strand", "+")
    for r in nmd:
        v = r.get("variant", {})
        m = re.match(r"p\.([A-Za-z]{3})\d+(\*|Ter)", str(v.get("hgvs_p", "")))
        if not m or len(str(v.get("ref"))) != 1 or len(str(v.get("alt"))) != 1:
            continue
        plus4 = minus1 = None
        note = ""
        cands = rt.candidate_stop_codons(m.group(1), v["ref"], v["alt"], strand)
        if network and len(cands) == 1:
            try:
                coord = rt.plus4_coordinate(int(v["pos"]), cands[0]["index"], strand)
                plus4 = rt.fetch_base(str(v["chrom"]), coord)
                minus1 = rt.fetch_base(str(v["chrom"]), rt.minus1_coordinate(int(v["pos"]), cands[0]["index"], strand))
            except Exception as e:                      # network or API failure: report, do not guess
                plus4 = minus1 = None
                note = f"+4 base not retrieved ({type(e).__name__})"
        else:
            note = "+4 base not retrieved (network disabled or ambiguous codon)"
        res = rt.assess(m.group(1), v["ref"], v["alt"], strand, bool(r.get("predicted_nmd")), plus4, minus1)
        if plus4 is None:
            res["note"] = note
        return res
    return None


def figure_tree(tree: dict, interventions: list[dict], out: Path, title: str, color="#1f4e8c"):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyBboxPatch
    nodes = tree["nodes"]
    level = {tree["top"]: 0}
    order = [tree["top"]]
    for n in order:
        for c in nodes[n].get("children", []):
            level[c] = level[n] + 1
            order.append(c)
    leaves = [n for n in order if nodes[n]["type"] == "BASIC"]
    xs = {n: i for i, n in enumerate(leaves)}
    def xpos(n):
        if n in xs:
            return xs[n]
        v = [xpos(c) for c in nodes[n]["children"]]
        xs[n] = sum(v) / len(v)
        return xs[n]
    for n in order:
        xpos(n)
    depth = max(level.values())
    fig, ax = plt.subplots(figsize=(7.2, 1.9 + 1.25 * depth))
    for n in order:
        for c in nodes[n].get("children", []):
            ax.plot([xs[n], xs[c]], [-level[n], -level[c]], color="#8a93a6", lw=0.9, zorder=1)
    for n in order:
        kind = nodes[n]["type"]
        w = 0.82
        ax.add_patch(FancyBboxPatch((xs[n] - w / 2, -level[n] - 0.2), w, 0.4, boxstyle="round,pad=0.02",
                                    fc="white" if kind != "BASIC" else "#e6ecf7", ec=color, lw=1, zorder=2))
        lab = nodes[n]["label"]
        ax.text(xs[n], -level[n] + 0.0, "\n".join(_wrap(lab, 22)), ha="center", va="center", fontsize=5.8, zorder=3)
        if kind in ("AND", "OR"):
            ax.text(xs[n], -level[n] - 0.33, kind, ha="center", va="center", fontsize=6.5, color="#b3261e",
                    fontweight="bold", zorder=3)
    for it in interventions:
        for e in it["addresses"]:
            if e in xs and e in leaves:
                ax.text(xs[e], -level[e] - 0.62, it["id"], ha="center", va="center", fontsize=7, color="white",
                        bbox=dict(boxstyle="round,pad=0.25", fc="#b3261e", ec="none"), zorder=3)
    ax.set_xlim(-0.7, len(leaves) - 0.3); ax.set_ylim(-depth - 0.9, 0.5)
    ax.set_title(title, loc="left", fontsize=8.5)
    ax.axis("off"); fig.tight_layout(); fig.savefig(out); plt.close(fig)


def _wrap(s: str, n: int) -> list[str]:
    words, line, out = s.split(), "", []
    for w in words:
        if len(line) + len(w) + 1 > n and line:
            out.append(line); line = w
        else:
            line = (line + " " + w).strip()
    return out + [line]


def figure_benefit_risk(ranking: list[dict], excluded: list[dict], out: Path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6.2, 3.8))
    ax.axhspan(fmea.S_STOP - 0.5, 10.5, color="#f6d6d3", alpha=0.6, lw=0)
    ax.axvspan(0.5, fmea.E_MIN - 0.5, color="#e6e6e6", alpha=0.6, lw=0)
    for r in excluded:
        ax.plot(r["E"], r["S"], "x", color="#6b6b6b", ms=6)
        ax.text(r["E"] + 0.06, r["S"] + 0.12, r["id"], fontsize=6.5, color="#6b6b6b")
    for r in ranking:
        ax.scatter(r["E"], r["S"], s=60 + 420 * r["coverage"], color="#1f4e8c", alpha=0.8, edgecolor="#1f2a44")
        ax.text(r["E"] + 0.1, r["S"] + 0.2, f"{r['id']}  (RPN {r['RPN']})", fontsize=7)
    ax.set_xlim(0.5, 5.5); ax.set_ylim(0.5, 10.5)
    ax.set_xlabel("Evidence grade E (1 = speculative, 5 = randomised trial)")
    ax.set_ylabel("Severity S if the hypothesis is wrong")
    ax.text(0.6, 10.1, "hard stop: severity", fontsize=7, color="#b3261e")
    ax.text(0.6, 1.0, "hard stop: evidence", fontsize=7, color="#555", rotation=90)
    ax.set_title("Candidates (circles; area = coverage of failure paths) and exclusions (crosses)", loc="left", fontsize=8)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    fig.tight_layout(); fig.savefig(out); plt.close(fig)


def figure_sim(simres: dict, out: Path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    names = list(simres["interventions"])[::-1]
    pb = [simres["interventions"][n]["p_better"] for n in names]
    pn = [simres["interventions"][n]["p_no_change"] for n in names]
    pw = [simres["interventions"][n]["p_worse"] for n in names]
    fig, ax = plt.subplots(figsize=(7.2, 3.0))
    y = range(len(names))
    ax.barh(y, pb, color="#2e7d4f", label="fewer errors")
    ax.barh(y, pn, left=pb, color="#cfd4dc", label="no change")
    ax.barh(y, pw, left=[a + b for a, b in zip(pb, pn)], color="#b3261e", label="more errors")
    ax.set_yticks(list(y)); ax.set_yticklabels(names, fontsize=7)
    ax.set_xlim(0, 1); ax.set_xlabel("share of 5,000 random parameter sets (toy model, not fitted)", fontsize=7)
    ax.tick_params(axis="x", labelsize=7); ax.legend(fontsize=7, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.2), frameon=False)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    fig.tight_layout(); fig.savefig(out); plt.close(fig)


def run(kb_path=None, outdir=None, network: bool = True) -> dict:
    kb = load_kb(kb_path)
    res = analyse(kb)
    out = Path(outdir or os.environ.get("MVA_WORKDIR") or "work_track2"); out.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(res["ranking"]).to_csv(out / "track2_ranking.csv", index=False)
    pd.DataFrame(res["excluded"]).to_csv(out / "track2_excluded.csv", index=False)
    rtk = readthrough_check(out, network)
    if rtk:
        res["readthrough_check"] = rtk
    simres = sim.run()
    raw = simres.pop("_raw")
    simres["dose_response_median_misseg"] = sim.dose_response(raw["params"])
    res["checkpoint_simulation"] = simres
    figure_sim(simres, out / "fig_t2_checkpoint_sim.pdf")
    figure_tree(kb["root_tree"], kb["interventions"], out / "fig_t2_root_tree.pdf", "Root-cause fault tree (AND of two alleles)")
    figure_tree(kb["consequence_tree"], kb["interventions"], out / "fig_t2_consequence_tree.pdf", "Consequence tree", "#6b4d9b")
    figure_benefit_risk(res["ranking"], [e for e in res["excluded"] if "E" in e and "S" in e], out / "fig_t2_benefit_risk.pdf")
    with open(out / "track2_summary.json", "w") as fh:
        json.dump(res, fh, indent=1, default=str)
    return res


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--kb"); ap.add_argument("--out"); ap.add_argument("--no-network", action="store_true")
    a = ap.parse_args(argv)
    res = run(a.kb, a.out, not a.no_network)
    print(json.dumps({k: res[k] for k in ("minimal_cut_sets", "fully_mitigating_sets_root")}, indent=1))
    print(pd.DataFrame(res["ranking"]).to_string(index=False))
    print("\nExcluded:"); print(pd.DataFrame(res["excluded"])[["id", "label", "stop"]].to_string(index=False))
    print("\nRank stability:", json.dumps(res["rank_stability"], indent=1))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile knowledge/bub1b_mva1_kb.json
{
 "_comment": "Curated knowledge for the BUB1B / MVA1 genotype class. Scores are expert judgements with rationale; every entry carries references (keys into 'references'). Hypotheses for follow-up, not treatment advice.",
 "root_tree": {
  "top": "R_TOP",
  "nodes": {
   "R_TOP": {
    "type": "AND",
    "label": "Insufficient BUBR1 function (biallelic)",
    "children": [
     "A1",
     "A2"
    ]
   },
   "A1": {
    "type": "OR",
    "label": "Nonsense allele (V1) provides no functional BUBR1",
    "children": [
     "B1",
     "B2"
    ]
   },
   "B1": {
    "type": "BASIC",
    "label": "Transcript removed by nonsense-mediated decay"
   },
   "B2": {
    "type": "BASIC",
    "label": "Truncated product lacks the pseudokinase domain and PP2A-B56 recruitment"
   },
   "A2": {
    "type": "OR",
    "label": "Missense allele (V2) provides no sufficient BUBR1",
    "children": [
     "B3",
     "B4"
    ]
   },
   "B3": {
    "type": "BASIC",
    "label": "Destabilised pseudokinase fold lowers protein level"
   },
   "B4": {
    "type": "BASIC",
    "label": "Reduced hyperphosphorylation and kinetochore PP2A-B56 function (inferred from the domain)"
   }
  }
 },
 "consequence_tree": {
  "top": "C_TOP",
  "nodes": {
   "C_TOP": {
    "type": "OR",
    "label": "Systemic harm from chronic chromosomal instability",
    "children": [
     "Cq1",
     "Cq2",
     "Cq3",
     "Cq4"
    ]
   },
   "Cq1": {
    "type": "BASIC",
    "label": "Tumour initiation (cancer predisposition)"
   },
   "Cq2": {
    "type": "BASIC",
    "label": "Micronuclei, cGAS-STING and type I interferon signalling"
   },
   "Cq3": {
    "type": "BASIC",
    "label": "Cellular senescence and inflammatory secretome (SASP)"
   },
   "Cq4": {
    "type": "BASIC",
    "label": "Apoptosis of progenitors, growth failure and microcephaly"
   }
  }
 },
 "interventions": [
  {
   "id": "I1",
   "label": "NAD+ precursor (niacin or nicotinamide riboside)",
   "drugs": [
    "niacin",
    "nicotinamide riboside"
   ],
   "tier": "root",
   "addresses": [
    "B3"
   ],
   "approval": "supplement",
   "E": 2,
   "S": 4,
   "D": 3,
   "gating_biomarker": "BUBR1 protein level in patient-derived cells; blood NAD+",
   "rationale": "SIRT2 raises BubR1 abundance in mice (North 2014), which is the mechanism relevant to a destabilised missense allele whose main measured defect is a lower protein level. No data in MVA patients or cells. Nicotinamide itself inhibits sirtuins at high dose and is avoided. Niacin is an approved medicine, nicotinamide riboside is a supplement. Severity includes the theoretical concern of supporting tumour growth.",
   "refs": [
    "north2014",
    "pseudokinase2019"
   ],
   "tags": [
    "cancer_risk"
   ]
  },
  {
   "id": "I2",
   "label": "JAK1/2 inhibitor (baricitinib), only if an interferon signature is present",
   "drugs": [
    "baricitinib"
   ],
   "tier": "consequence",
   "addresses": [
    "Cq2"
   ],
   "approval": "approved_paediatric",
   "E": 2,
   "S": 8,
   "D": 2,
   "gating_biomarker": "blood interferon-stimulated gene score",
   "rationale": "Chromosome missegregation produces micronuclei that activate cGAS-STING; baricitinib has been used in paediatric interferonopathies. No evidence yet of an interferon signature in MVA1, so use is conditional on measuring it. Severity is high because of infection risk and the class warning on malignancy in a child with cancer predisposition.",
   "refs": [
    "mackenzie2017",
    "harding2017",
    "bakhoum2018",
    "sanchez2018"
   ],
   "tags": [
    "cancer_risk",
    "infection"
   ]
  },
  {
   "id": "I3",
   "label": "Intermittent dasatinib (senolytic), alone or with quercetin",
   "drugs": [
    "dasatinib",
    "quercetin"
   ],
   "tier": "consequence",
   "addresses": [
    "Cq3"
   ],
   "approval": "approved_paediatric",
   "E": 3,
   "S": 7,
   "D": 4,
   "gating_biomarker": "p16/p21 and SASP markers (imperfect)",
   "rationale": "In BubR1-insufficient progeroid mice, genetic clearance of p16-positive cells delays age-related pathology, which is in vivo evidence in the same gene. Dasatinib itself was not tested in that model. Haematological and pleural toxicity matter in a child with a fragile marrow.",
   "refs": [
    "baker2011",
    "baker2016",
    "xu2018"
   ],
   "tags": [
    "marrow"
   ]
  },
  {
   "id": "I4",
   "label": "Translational readthrough (gentamicin) for the nonsense allele",
   "drugs": [
    "gentamicin"
   ],
   "tier": "root",
   "addresses": [
    "B2"
   ],
   "approval": "approved_off_label",
   "E": 2,
   "S": 7,
   "D": 5,
   "gating_biomarker": "full-length BUBR1 by immunoblot in patient-derived cells",
   "rationale": "The nonsense allele creates a premature stop codon whose identity and local context are determined per patient (readthrough check; UGA has the highest basal readthrough), but NMD is predicted (B1 remains), so clearing B2 alone cannot restore the allele. Ataluren is not available in the EU (marketing authorisation not renewed, expired 28 March 2025) and is not FDA-approved. Aminoglycosides are oto- and nephrotoxic. Gentamicin response depends on local sequence context (a U at -1 is the main determinant).",
   "refs": [
    "floquet2012",
    "ema_ataluren"
   ],
   "tags": [
    "ototoxic"
   ]
  },
  {
   "id": "I5",
   "label": "Type I interferon receptor blockade (anifrolumab), only if an interferon signature is present",
   "drugs": [
    "anifrolumab"
   ],
   "tier": "consequence",
   "addresses": [
    "Cq2"
   ],
   "approval": "approved_adult",
   "E": 2,
   "S": 7,
   "D": 2,
   "tags": [
    "cancer_risk",
    "infection"
   ],
   "gating_biomarker": "blood interferon-stimulated gene score",
   "rationale": "Found by the database screen, not by the manual review. Monoclonal antibody against IFNAR1: blocks type I interferon signalling at the receptor, narrower than JAK1/2 inhibition. Approved for adult lupus; paediatric approval was not found, so any use in a child would be off-label. No evidence in MVA. Severity is provisional (7): infection risk and the class caution for immunosuppression in a cancer-prone child; one point below baricitinib because it is narrower, which is a judgement.",
   "refs": [
    "ema_saphnelo",
    "mackenzie2017",
    "harding2017"
   ]
  }
 ],
 "exclusions": [
  {
   "id": "X1",
   "label": "MPS1/TTK inhibitors",
   "class": "checkpoint abrogation",
   "S": 10,
   "E": 2,
   "approval": "none",
   "reason": "Abrogate the spindle checkpoint in every cell, and in this genotype every cell is already compromised; no healthy population to spare. Relevant only to a tumour, never as systemic therapy. Not approved."
  },
  {
   "id": "X2",
   "label": "Microtubule agents (benzimidazoles, taxanes, vinca alkaloids)",
   "class": "spindle poisons",
   "S": 9,
   "E": 2,
   "approval": "approved_adult",
   "reason": "Perturb the spindle and raise the chance of mis-segregation. A weakened checkpoint allows slippage, so cells escape arrest with damage. The 'speed bump' idea is backwards.",
   "opposite_mechanism": "perturbs the spindle"
  },
  {
   "id": "X3",
   "label": "HDAC inhibitors",
   "class": "epigenetic",
   "S": 8,
   "E": 1,
   "approval": "approved_adult",
   "reason": "Histone hyperacetylation in mitosis has been reported to cause segregation defects; no support for strengthening the centromere."
  },
  {
   "id": "X4",
   "label": "ATR/CHK1 inhibitors",
   "class": "checkpoint abrogation",
   "S": 10,
   "E": 2,
   "approval": "none",
   "reason": "Force mitotic entry with damage, which would be toxic to normal tissue; not approved."
  },
  {
   "id": "X5",
   "label": "Allosteric activators of kinesins",
   "class": "bypass",
   "S": 3,
   "E": 1,
   "approval": "none",
   "reason": "No approved agent exists."
  },
  {
   "id": "X6",
   "label": "Fatty-acid-synthesis inhibition to starve aneuploid cells",
   "class": "metabolic",
   "S": 6,
   "E": 1,
   "approval": "none",
   "reason": "Selective killing of aneuploid cells has not been shown in vivo and a systemic approach is unsafe in a child."
  },
  {
   "id": "X7",
   "label": "Direct cGAS or STING inhibitors",
   "class": "innate immunity",
   "S": 4,
   "E": 1,
   "approval": "none",
   "reason": "No approved agent; the approved route into the pathway is JAK inhibition (I2)."
  },
  {
   "id": "X8",
   "label": "Nicotinamide as an NAD+ source",
   "class": "NAD+",
   "S": 4,
   "E": 2,
   "approval": "approved_adult",
   "reason": "At high dose it inhibits sirtuins, the opposite of the intended effect; niacin or nicotinamide riboside are preferred.",
   "opposite_mechanism": "inhibits sirtuins at high dose"
  },
  {
   "id": "X9",
   "label": "AMPK activation (metformin) for micronucleus clearance",
   "class": "autophagy",
   "S": 2,
   "E": 1,
   "approval": "approved_paediatric",
   "reason": "Weak, indirect link to this mechanism."
  }
 ],
 "references": {
  "hanks2004": "Hanks S, et al. Constitutional aneuploidy and cancer predisposition caused by biallelic mutations in BUB1B. Nat Genet 2004;36:1159-61.",
  "baker2004": "Baker DJ, et al. BubR1 insufficiency causes early onset of aging-associated phenotypes and infertility in mice. Nat Genet 2004;36:744-9.",
  "baker2011": "Baker DJ, et al. Clearance of p16Ink4a-positive senescent cells delays ageing-associated disorders. Nature 2011;479:232-6.",
  "baker2016": "Baker DJ, et al. Naturally occurring p16Ink4a-positive cells shorten healthy lifespan. Nature 2016;530:184-9.",
  "north2014": "North BJ, et al. SIRT2 induces the checkpoint kinase BubR1 to increase lifespan. EMBO J 2014;33:1438-53.",
  "pseudokinase2019": "The BUBR1 pseudokinase domain promotes efficient kinetochore PP2A-B56 recruitment to regulate spindle checkpoint silencing and chromosome alignment. bioRxiv 2019, doi:10.1101/733378 (preprint, not peer reviewed in the version consulted).",
  "mackenzie2017": "Mackenzie KJ, et al. cGAS surveillance of micronuclei links genome instability to innate immunity. Nature 2017;548:461-5.",
  "harding2017": "Harding SM, et al. Mitotic progression following DNA damage enables pattern recognition within micronuclei. Nature 2017;548:466-70.",
  "bakhoum2018": "Bakhoum SF, et al. Chromosomal instability drives metastasis through a cytosolic DNA response. Nature 2018;553:467-72.",
  "sanchez2018": "Montealegre Sanchez GA, et al. JAK1/2 inhibition with baricitinib in the treatment of autoinflammatory interferonopathies. J Clin Invest 2018;128:3041-52.",
  "xu2018": "Xu M, et al. Senolytics improve physical function and increase lifespan in old age. Nat Med 2018;24:1246-56.",
  "floquet2012": "Floquet C, et al. Statistical analysis of readthrough levels for nonsense mutations in mammalian cells reveals a major determinant of response to gentamicin. PLoS Genet 2012;8:e1002608.",
  "ema_ataluren": "European Medicines Agency. Translarna (ataluren): European Public Assessment Report page; conditional marketing authorisation not renewed, expired 28 March 2025 (European Commission decision). Accessed 14 October 2026.",
  "cimini2003": "Cimini D, et al. Histone hyperacetylation in mitosis prevents sister chromatid separation and produces chromosome segregation defects. Mol Biol Cell 2003;14:3821-33.",
  "ema_saphnelo": "European Medicines Agency. Saphnelo (anifrolumab), EPAR page (indication: adults with systemic lupus erythematosus). Accessed 23 October 2026."
 },
 "uncertainty": {
  "relevance_note": "r = P(the BUB1B pair is the disease-causing compound heterozygous genotype) = P(alleles in trans) x P(missense allele is hypomorphic). Root-tier interventions only help if it is true; consequence-tier ones act on harm that is present whatever the cause (and are gated by a biomarker).",
  "r_grid": [
   0.05,
   0.1,
   0.15,
   0.2,
   0.3,
   0.4,
   0.5,
   0.6,
   0.7,
   0.8,
   0.9,
   1.0
  ],
  "modifier": {
   "label": "additional heterozygous loss-of-function variant in a tumour-predisposition gene (identified in the same genome)",
   "severity_delta": {
    "cancer_risk": 1
   }
  }
 },
 "screen_review": {
  "PACLITAXEL PROTEIN-BOUND": "microtubule agent (class X2); the BCL2 record is incidental",
  "TIRBANIBULIN": "inhibits tubulin polymerisation as well as Src (class X2)",
  "OBLIMERSEN": "BCL2 antisense; withdrawn or not marketed, treat as not available",
  "VANDETANIB": "oncology kinase inhibitor with many targets; Src record not specific",
  "BOSUTINIB": "oncology kinase inhibitor; same class concerns as dasatinib",
  "IBRUTINIB": "oncology BTK inhibitor; Src record secondary",
  "ACALABRUTINIB": "oncology BTK inhibitor; Src record secondary",
  "VENETOCLAX": "BCL2 inhibitor, oncology; neutropenia; senolytic use only experimental",
  "NAVITOCLAX": "BCL2/BCL-xL inhibitor; thrombocytopenia; not approved in most settings",
  "AMLEXANOX": "TBK1 inhibitor; approval is regional (not scored)",
  "LESTAURTINIB": "investigational; approval flag questionable",
  "REPOTRECTINIB": "oncology kinase inhibitor; JAK2 record secondary"
 }
}

In [ ]:
%%writefile knowledge/TEMPLATE_new_genotype.json
{
 "_comment": "SYNTHETIC TEMPLATE to show the schema; not a real disease. Copy, replace the labels, evidence grades, severities, detection scores and references, and run: python -m mvapipe.track2.run --kb knowledge/your_file.json",
 "root_tree": {"top": "R_TOP", "nodes": {
  "R_TOP": {"type": "AND", "label": "Loss of function of gene X (biallelic)", "children": ["A1", "A2"]},
  "A1": {"type": "OR", "label": "Allele 1 non-functional", "children": ["B1", "B2"]},
  "A2": {"type": "OR", "label": "Allele 2 non-functional", "children": ["B3", "B4"]},
  "B1": {"type": "BASIC", "label": "Mechanism 1 of allele 1"}, "B2": {"type": "BASIC", "label": "Mechanism 2 of allele 1"},
  "B3": {"type": "BASIC", "label": "Mechanism 1 of allele 2"}, "B4": {"type": "BASIC", "label": "Mechanism 2 of allele 2"}}},
 "consequence_tree": {"top": "C_TOP", "nodes": {
  "C_TOP": {"type": "OR", "label": "Downstream harm", "children": ["Cq1", "Cq2"]},
  "Cq1": {"type": "BASIC", "label": "Consequence 1"}, "Cq2": {"type": "BASIC", "label": "Consequence 2"}}},
 "interventions": [
  {"id": "I1", "label": "Example agent A", "drugs": ["A"], "tier": "root", "addresses": ["B3"], "approval": "approved_adult",
   "E": 2, "S": 3, "D": 3, "gating_biomarker": "example", "rationale": "placeholder", "refs": []},
  {"id": "I2", "label": "Example agent B", "drugs": ["B"], "tier": "consequence", "addresses": ["Cq1"], "approval": "approved_paediatric",
   "E": 3, "S": 5, "D": 4, "gating_biomarker": "example", "rationale": "placeholder", "refs": []}],
 "exclusions": [
  {"id": "X1", "label": "Example rejected idea", "class": "example", "S": 9, "E": 1, "approval": "none", "reason": "placeholder"}],
 "references": {}
}

## 1. Fault trees and minimal cut sets
The root tree is AND(allele 1 fails, allele 2 fails), each allele an OR of mechanisms. The consequence tree is an OR of downstream harms.

In [ ]:
import pandas as pd
from mvapipe.track2 import run as t2, faulttree as ft
kb = t2.load_kb("knowledge/bub1b_mva1_kb.json")
res = t2.analyse(kb)
print("root cut sets:", res["minimal_cut_sets"]["root"])
print("smallest sets that break every root cut set:", res["fully_mitigating_sets_root"])

## 2. FMEA ranking (lower RPN = better) and hard stops

In [ ]:
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 70)
display(pd.DataFrame(res["ranking"]))
display(pd.DataFrame(res["excluded"])[["id", "label", "stop"]])

## 3. Rank stability
Every S, O and D is perturbed by -1, 0 or +1 (10,000 draws). Reports how often the order or the first place survives.

In [ ]:
print(json.dumps(res["rank_stability"], indent=1))

## 4. Figures and files

In [ ]:
TRACK1_WORK = os.environ.get("MVA_WORKDIR", "work_track2")   # point to the Track 1 work directory to enable the readthrough step
res = t2.run(outdir=TRACK1_WORK, network=True)
print(json.dumps(res.get("readthrough_check", "no Track 1 outputs found: readthrough step skipped"), indent=1))
print(sorted(os.listdir(TRACK1_WORK)))

## 5. Genotype-uncertainty scenarios
r = probability that the compound-heterozygous genotype is the causal one. Root-tier candidates lose value as r falls; consequence-tier ones do not. A second tumour-predisposition variant raises severity for tagged interventions.

In [ ]:
print(json.dumps(res["scenarios"], indent=1, default=str))

## 6. Spindle-checkpoint simulation (toy model, direction of effect only)
Not fitted to data. Reports the share of 5,000 random parameter sets in which each mitosis-acting intervention gives fewer, equal or more uncorrected chromosome errors in a checkpoint-weakened cell.

In [ ]:
print(json.dumps(res["checkpoint_simulation"], indent=1, default=str)[:3500])

## 7. Database and pathway screen (needs Internet; only gene symbols are sent)
DGIdb, Open Targets and Enrichr for the mechanism's gene sets. Failures are listed, not hidden. Paste the printed table back for review.

In [ ]:
import subprocess; print(subprocess.run([sys.executable, "-m", "mvapipe.track2.db_screen", "--out", "work_track2"], capture_output=True, text=True).stdout[-6000:])

## 8. Same engine, a different genotype class
The template knowledge file is synthetic; it shows that a new disease needs a new JSON file, not new code.

In [ ]:
kb2 = t2.load_kb("knowledge/TEMPLATE_new_genotype.json")
display(pd.DataFrame(t2.analyse(kb2)["ranking"]))